# NB09 — Label budget: how many local patients are needed?

**Purpose.** Repeatedly subsample n local patients (n = 25 … 800) from each hospital's 2014 data, update the model with them, and evaluate on that hospital's 2015 data. Two scenarios: a **stand-alone** hospital (only its own n patients: local intercept / logistic / isotonic) and a **network** hospital (global recalibration from other hospitals + its own n patients combined by empirical-Bayes shrinkage). Global quantities are computed leave-this-hospital-out, so no hospital sees its own data twice.

**Research question.** RQ5 — minimum local sample for acceptable calibration

| | |
|---|---|
| **Inputs** | `models/pred_eicu.parquet` |
| **Outputs** | Table 7, Figure 7 |
| **Free-Colab runtime** | 20–40 min (checkpointed per repeat) |

**How to run:** `Runtime → Run all`. If Colab disconnects, just run all again — finished steps are cached.
At the end, download the zip printed in the last cell into your local `Plan A/v4/results/` folder.

In [ ]:
# ==== Setup (identical in every notebook) ====
import os, sys
if 'google.colab' in sys.modules:
    from google.colab import drive
    drive.mount('/content/drive')
    os.environ.setdefault('TRANSPORT_BASE', '/content/drive/MyDrive/Plan A/v4')
BASE = os.environ['TRANSPORT_BASE']
sys.path.insert(0, os.path.join(BASE, 'code'))
import importlib, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import transport_utils as tu
importlib.reload(tu)
warnings.filterwarnings('ignore')
tu.set_style()
OUT = tu.init_notebook('NB09')

In [ ]:
m, e = tu.load_analysis()
pred = pd.read_parquet(os.path.join(tu.MODELS, 'pred_eicu.parquet'))
upd = pred[pred['split'] == 'update'].reset_index(drop=True)
ev = pred[pred['split'] == 'eval'].reset_index(drop=True)
R = 3 if tu.FAST else 50
PCOL = 'p_lgbm'
cnt = ev.groupby('hospital_id').agg(n=('mortality', 'size'), d=('mortality', 'sum'))
nupd = upd.groupby('hospital_id').size()
ev_ok = cnt[(cnt.n >= tu.MIN_HOSP_N) & (cnt.d >= tu.MIN_HOSP_EVENTS)].index
# FIXED hospital set: every hospital contributes at every n (otherwise the curve mixes
# different hospitals at different n). Largest n is 400 if >= 30 hospitals qualify, else 200.
NMAX = 400 if sum(nupd.get(h, 0) >= 400 for h in ev_ok) >= 30 else 200
if tu.FAST:
    NMAX = 100
NS = [n for n in [25, 50, 100, 200, 400] if n <= NMAX]
HOSP = [h for h in ev_ok if nupd.get(h, 0) >= NMAX]
print(f'fixed hospital set: {len(HOSP)} hospitals with >= {NMAX} update patients | n grid {NS} | repeats {R}')

In [ ]:
# Leave-one-hospital-out network quantities (global logistic + EB prior) -- computed once
site_int = {}
for h, g in upd.groupby('hospital_id'):
    site_int[h] = g
loo = {}
for h in HOSP:
    other = upd[upd['hospital_id'] != h]
    ag, bg = tu.fit_logistic(other[PCOL].values, other['mortality'].values)
    pg = tu.apply_logistic(other[PCOL].values, ag, bg)
    t = []
    for hh, g in other.assign(pg=pg).groupby('hospital_id'):
        a, se = tu.fit_intercept(g['pg'].values, g['mortality'].values); t.append((a, se))
    t = np.array(t)
    ma = tu.meta_analysis(t[:, 0], t[:, 1], hksj=False)
    loo[h] = dict(ag=ag, bg=bg, mu=ma['mu'], tau2=ma['tau2'])
ev_by_h = {h: g for h, g in ev.groupby('hospital_id') if h in HOSP}
upd_by_h = {h: g for h, g in upd.groupby('hospital_id') if h in HOSP}
print('network priors ready; median tau =', np.median([np.sqrt(v['tau2']) for v in loo.values()]).round(3))

In [ ]:
def eval_h(y, p):
    o, ex = y.sum(), p.sum()
    oe = o / ex
    cs = tu.calibration_intercept_slope(y, p)
    return oe, abs(np.log(oe)) if o > 0 else np.nan, cs['slope']

def baselines():
    rows = []
    for h in HOSP:
        g = ev_by_h[h]; y = g['mortality'].values; p = g[PCOL].values
        L = loo[h]
        for name, pp in [('No recalibration', p), ('Global logistic (network, no local data)',
                                                    tu.apply_logistic(p, L['ag'], L['bg']))]:
            oe, a, sl = eval_h(y, pp)
            rows.append(dict(method=name, hospital_id=h, oe=oe, abs_ln_oe=a, slope=sl))
    return pd.DataFrame(rows)

def one_repeat(r):
    rng = np.random.default_rng(1000 + r)
    rows = []
    for h in HOSP:
        gu, ge = upd_by_h[h], ev_by_h[h]
        ye, pe = ge['mortality'].values, ge[PCOL].values
        L = loo[h]
        pe_g = tu.apply_logistic(pe, L['ag'], L['bg'])
        for n in NS:
            if len(gu) < n:
                continue
            s = gu.iloc[rng.choice(len(gu), n, replace=False)]
            ys, ps = s['mortality'].values, s[PCOL].values
            preds = {}
            a, _ = tu.fit_intercept(ps, ys); preds['Local intercept'] = tu.apply_logistic(pe, a)
            aa, bb = tu.fit_logistic(ps, ys)
            if not (np.isfinite(aa) and np.isfinite(bb) and bb > 0):
                aa, bb = a, 1.0
            preds['Local logistic'] = tu.apply_logistic(pe, aa, bb)
            preds['Local isotonic'] = tu.fit_isotonic(ps, ys).transform(pe) if 0 < ys.sum() < n else tu.apply_logistic(pe, a)
            a_loc, se_loc = tu.fit_intercept(tu.apply_logistic(ps, L['ag'], L['bg']), ys)
            a_eb = tu.eb_shrink_single(a_loc, se_loc, L['mu'], L['tau2'])
            preds['Network + EB local intercept'] = tu.expit(tu.logit(pe_g) + a_eb)
            for k, pp in preds.items():
                oe, ab, sl = eval_h(ye, pp)
                rows.append(dict(rep=r, n=n, events_in_sample=int(ys.sum()), method=k, hospital_id=h,
                                 oe=oe, abs_ln_oe=ab, slope=sl))
    return pd.DataFrame(rows)

base = tu.cached('baselines', baselines)
reps = []
for r in range(R):
    reps.append(tu.cached(f'rep_{r:03d}', one_repeat, r))
    if (r + 1) % 5 == 0 or r == R - 1:
        print(f'  repeat {r+1}/{R} done', flush=True)
lb = pd.concat(reps, ignore_index=True)

In [ ]:
lb['acceptable'] = lb['oe'].between(*tu.OE_ACCEPT)
base['acceptable'] = base['oe'].between(*tu.OE_ACCEPT)
agg = lb.groupby(['method', 'n']).agg(hospitals=('hospital_id', 'nunique'),
                                      median_events=('events_in_sample', 'median'),
                                      pct_acceptable=('acceptable', lambda s: 100 * s.mean()),
                                      median_abs_ln_oe=('abs_ln_oe', 'median'),
                                      q75_abs_ln_oe=('abs_ln_oe', lambda s: s.quantile(.75)),
                                      median_slope=('slope', 'median')).reset_index()
bagg = base.groupby('method').agg(pct_acceptable=('acceptable', lambda s: 100 * s.mean()),
                                  median_abs_ln_oe=('abs_ln_oe', 'median'), median_slope=('slope', 'median')).reset_index()
tu.save_table(agg.round(3), 'table7_label_budget')
tu.save_table(bagg.round(3), 'table7_baselines')
min_n = {}
for k, g in agg.groupby('method'):
    ok = g[g['pct_acceptable'] >= 80]
    min_n[k] = int(ok['n'].min()) if len(ok) else None
print('Minimum n reaching >=80% hospitals acceptable:', min_n)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
methods = ['Local intercept', 'Local logistic', 'Local isotonic', 'Network + EB local intercept']
for ax, met, ttl in [(axes[0], 'pct_acceptable', '% hospitals with acceptable O/E (0.8-1.2)'),
                     (axes[1], 'median_abs_ln_oe', 'Median |ln O/E| (lower is better)')]:
    for i, k in enumerate(methods):
        g = agg[agg['method'] == k]
        ax.plot(g['n'], g[met], marker='o', ms=5, color=tu.SERIES[i], label=k)
    for j, (_, r) in enumerate(bagg.iterrows()):
        ax.axhline(r[met], color=[tu.C['grey'], tu.C['ink2']][j % 2], lw=1, ls='--', label=r['method'])
    ax.set_xscale('log'); ax.set_xticks(NS); ax.set_xticklabels(NS)
    ax.set_xlabel('Local patients used for updating (n)'); ax.set_title(ttl)
axes[0].legend(loc='lower right', fontsize=7)
tu.save_fig(fig, 'fig7_label_budget'); plt.show()

tu.finish_notebook({'hospitals': len(HOSP), 'repeats': R, 'label_budget': agg.round(4).to_dict('records'),
                    'baselines': bagg.round(4).to_dict('records'), 'min_n_for_80pct_acceptable': min_n})